# Stage 2 + 3 on GPU — extrusion, polymer dynamics, cell cycle, mRNA decay

**Run order:** cells 1–3 set up, then 4 (Stage 2+3), 5 (cell cycle), 6 (mRNA decay). Each is
independent — if one fails the others still run.

**What each section actually needs.** Cell 4 needs only the repo plus a 0.9 MB CTCF BED, so it runs
on a bare Colab. Cell 5 needs the three single-cell h5ads (~3.3 GB) and skips without them. Cell 6
needs `nlz_K562_gwps.npz`, which cell 3b rebuilds from `gwps.h5ad` if Drive has only the latter.

**The bars, so a result can be judged rather than admired:**

| module | bar it must beat | where the bar came from |
|---|---|---|
| Stage 2+3 extrusion + Langevin | **AUPRC 0.663** | CTCF-count contact feature, a bisect over a BED file |
| cell cycle | housekeeping control must collapse | genes that do not cycle must not fit θ |
| mRNA decay | **R² 0.3266** | `halflife.py`, 69 sequence features vs SLAM-seq |

The analytic Rouse model already scored **0.6107–0.6162** against that 0.663 — real physics, beaten
by counting. Stage 2+3 exists to check whether excluded volume, non-equilibrium motor loading, and a
distribution-shape observable rather than ⟨R²⟩ change that. (The observable is ⟨d⁻³⟩, not the hard
threshold P(d<d_c) originally planned — that turned out to be a rare event, median exactly 0.00000
even with a properly relaxed chain. See section 4.) It is a fair shot at a bar already set high.

## 1. GPU check

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'no nvidia-smi')
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('WARNING: no GPU. Runtime > Change runtime type > A100.')

## 2. Clone

In [ ]:
REPO = 'nikku03/cell'
BRANCH = 'claude/vectorize-gex-propensity-zp09w8'
import os, shutil, subprocess

def clone(token=None):
    if os.path.isdir('/content/cell'):
        shutil.rmtree('/content/cell')
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, '/content/cell'],
                       capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).replace(token or '\0', '***')

rc, out = clone()
print(out)
if rc != 0:
    from getpass import getpass
    tok = getpass('GitHub token (hidden, Enter to skip): ').strip()
    if tok:
        rc, out = clone(tok); print(out)
if rc != 0:
    raise SystemExit('clone failed')
os.chdir('/content/cell')
print(subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout)

import sys, time

def run_streaming(script, env_extra=None):
    """Run a repo script, printing each line as it arrives. Returns the exit code."""
    env = dict(os.environ)
    env.update(env_extra or {})
    t0 = time.time()
    p = subprocess.Popen([sys.executable, '-u', script], cwd='/content/cell',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    print(f'\n[{script}] exit {p.returncode} in {time.time()-t0:.0f}s')
    return p.returncode

## 3. Data

The CRISPR feature matrix, the TF compendium and the CTCF peaks all ship in the repo or fetch in
seconds. The single-cell h5ads (RPE1 / Jurkat / HepG2, ~1 GB each) are only needed by the cell-cycle
cell and are looked for on Drive — if absent, that cell skips rather than failing.

In [ ]:
import os, subprocess, fnmatch
SCRATCH = '/content/data'
os.makedirs(SCRATCH, exist_ok=True)
os.environ['CELL_SCRATCH'] = SCRATCH
os.environ['CELL_OUT'] = '/content/cell/outputs/orphan'

# CTCF peaks: the only external fetch Stage 2+3 needs, ~0.9 MB
ctcf = f'{SCRATCH}/ctcf_gw.bed.gz'
if not os.path.exists(ctcf):
    u = 'https://www.encodeproject.org/files/ENCFF519CXF/@@download/ENCFF519CXF.bed.gz'
    subprocess.run(['curl','-sSL','-o',ctcf,u])
print('CTCF peaks:', os.path.getsize(ctcf)//1000, 'KB')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
except Exception as e:
    DRIVE = None
    print('Drive not mounted:', type(e).__name__)

# link single-cell sources if they are on Drive -- symlink, never copy a 1 GB file
# nlz_K562_gwps.npz is what the mRNA-export signature test measures on; the h5ads are the
# cell-cycle inputs. Anything missing makes its own section skip with a message, not crash.
WANT = ('rpe1.h5ad','nadig_jurkat.h5ad','nadig_hepg2.h5ad','gwps.h5ad','nlz_K562_gwps.npz')
found = {}
if DRIVE:
    for root, dirs, files in os.walk(DRIVE):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for fn in files:
            if fn in WANT and fn not in found:
                found[fn] = os.path.join(root, fn)
            elif fnmatch.fnmatch(fn, '*gwps*bulk*.h5ad') and 'gwps.h5ad' not in found:
                found['gwps.h5ad'] = os.path.join(root, fn)
for k, v in found.items():
    d = f'{SCRATCH}/{k}'
    if not os.path.exists(d):
        os.symlink(v, d)
    print(f'  linked {k}  ({os.path.getsize(v)/1e6:.0f} MB)')
print('missing:', [w for w in WANT if w not in found] or 'none')

## 3b. Rebuild `nlz_K562_gwps.npz` if it is missing

Section 6's export-signature test needs the tide-removed K562 profile matrix. That file is **derived**
from `gwps.h5ad`, not downloaded — so if Drive has the h5ad but not the npz (the usual case, since the
npz is a build product), this rebuilds it in place. Reads the h5ad directly via `CELL_GWPS` rather than
copying 375 MB into scratch to satisfy a filename.

Skip this cell if cell 3 reported `nlz_K562_gwps.npz` as found.

In [ ]:
import os
npz = f"{SCRATCH}/nlz_K562_gwps.npz"
if os.path.exists(npz):
    print('already present:', os.path.getsize(npz)/1e6, 'MB')
elif os.path.exists(f'{SCRATCH}/gwps.h5ad'):
    os.environ['CELL_GWPS'] = f'{SCRATCH}/gwps.h5ad'
    run_streaming('colab/gwps_rebuild.py')
    print('built:', os.path.exists(npz))
else:
    print('gwps.h5ad is absent too -- section 6 will run its solver checks and skip the signature test.')

## 4. Stage 2 + 3 — KMC extrusion → batched Langevin → AUPRC vs **0.663**

Three knobs: `EXT_NSUB` (pairs simulated), `EXT_STEPS`, `EXT_REPLICA`. **The ETA prints within the
first minute** — if it is longer than you want, stop the cell and lower `EXT_NSUB`. Do not lower it
below ~800: the score is 5-fold CV grouped by chromosome, and with too few positives per fold the
AUPRC is undefined. It raises rather than printing a verdict off an empty average — a smoke run at
150 pairs returned `nan` for every arm and still printed "NO-GO", which is the failure that guard
exists to prevent.

**Arms, and what each one settles:**

| arm | question |
|---|---|
| `identity` | epigenetics + TF identity, no contact information |
| `count_contact` | **the bar** — CTCF peaks between the two coordinates, rescored on this subset |
| `mean_dist_only` | ⟨d⟩ — the simulation's version of what the closed form already gave |
| `SIM_only` | ⟨d⁻³⟩ — sensitive to the *shape* of the distance distribution, not just its 2nd moment |
| `SIM_plus_count` | both |

So `SIM_only − mean_dist_only` isolates whether distribution shape beats the second moment, which is
the only thing a simulation can offer over the analytic Rouse model that already scored 0.611–0.616.

Before any of that it prints `<d>/sqrt(separation)`. A Gaussian coil gives ~0.92 (the sandbox run
gives 1.52 — swollen by excluded volume, as a self-avoiding chain should be). An unrelaxed chain
gives tens, and the run aborts — because a chain that never left its initial condition produces
confident numbers that are pure artifact. The first version of this code initialised a straight rod
and every contact frequency came out exactly zero.

**A `GO` requires four things at once**, because each has produced a false positive here before: a
gain ≥ 0.01, a gain that beats a *shuffled-simulation* control (5 independent draws, not 1), a gain
larger than 2 se of that comparison, and a sign that survives changing the fold assignment. The
sandbox run at 400 pairs came out `+0.0226` raw — but the shuffled control alone gave `+0.0126`, so
net was `+0.0101 ± 0.0084`, z = 1.19, and the honest verdict was **UNDERPOWERED**, not GO.

Output **streams**: `capture_output=True` would show a blank cell for half an hour and be
indistinguishable from a hang.

In [ ]:
# Pairs bought at the expense of per-pair sampling. A 400-pair sandbox run returned
# net +0.0101 +/- 0.0084 -- the effect and the control's scatter were the same size, so the
# binding constraint is the NUMBER OF PAIRS, not the precision of each one. se falls as
# 1/sqrt(pairs); halving EXT_STEPS costs far less than that gains.
run_streaming('colab/extrude.py', {'EXT_NSUB': '3000', 'EXT_STEPS': '1200', 'EXT_REPLICA': '32'})

## 5. Cell cycle — continuous θ, C_vol(θ), f_i(θ)

Needs the single-cell h5ads from cell 3. Validates four things: θ spans the circle (the centred-atan2
fix), total UMI rises with θ, the Fourier fit separates cyclical genes from housekeeping **once
C_vol is divided out**, and knockouts shift the phase distribution — which is a confound in every
pseudobulk number this project has produced.

In [ ]:
run_streaming('colab/cellcycle.py')

## 6. mRNA decay — two-compartment model + export-signature test

CPU, seconds. Verifies the integrator against its analytic steady state, quantifies what an export
blockade does to the nuclear fraction, and tests whether nuclear-export knockouts share a response
signature against a ribosome-biogenesis control. The trans-factor gate (eCLIP / Ago-CLIP / m6A vs the
0.3266 sequence model) reports its inputs as absent rather than pretending to run.

In [ ]:
run_streaming('colab/mrna_decay.py')

## 8. Borzoi in-silico deletion — does frozen sequence grammar beat measured tracks?

For each gene, Borzoi predicts expression from 524 kb of sequence centred on its TSS. Delete a
candidate element's sequence, predict again, and the drop is that element's predicted contribution —
the CRISPRi experiment run in silico. **One scalar per pair**, which is the right size for 569
positives; trunk embeddings would be 3,072 columns and would fit noise.

**Receptive field decides the experiment.** An element must sit within ±262,144 bp of the TSS, which
is 3,960 of 10,331 pairs — but those hold **483 of 569 positives (84.9%)**, because positives
concentrate at short range. Subset base rate is 0.1220 vs 0.0551 overall, so **AUPRC here is not
comparable to the 0.6881 headline**; the baseline is rescored on exactly the same pairs.

**Two controls.** Shuffling the deletion scores across pairs tests for an extra-column effect.
The *sham* deletion removes an equal-sized span at the same distance on the other side of the TSS —
deleting any few hundred bp near a gene lowers predicted expression somewhat, and this asks whether
deleting *this* element lowers it more.

~9,146 forward passes (1,226 reference + 3,960 real + 3,960 sham). Set `BORZOI_NSUB` to subsample.

Everything except the forward pass was validated on CPU: TSS centring (offset exactly 0), one-hot
cleanliness, masking span, 0 mirror failures. The model's output layout was **not** — the cell prints
the output shape on its first batch so a wrong axis is visible immediately.

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','borzoi-pytorch','py2bit'])
tb = f'{SCRATCH}/hg38.2bit'
if not os.path.exists(tb):
    print('fetching hg38.2bit (~835 MB)...')
    subprocess.run(['curl','-sSL','-o',tb,'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.2bit'])
print('hg38.2bit', os.path.getsize(tb)//10**6, 'MB')
run_streaming('colab/borzoi_gate.py')

## 9. Save results to Drive

The sandbox this was developed in rolled back eleven times in one session. Anything not saved is
gone when the Colab VM recycles.

In [ ]:
import os, shutil, json
src = '/content/cell/outputs/orphan'
dst = '/content/drive/MyDrive/cell_results' if os.path.isdir('/content/drive/MyDrive') else None
keep = ['extrude_gate.json','cellcycle.json','mrna_decay.json','borzoi_gate.json']
if dst:
    os.makedirs(dst, exist_ok=True)
for f in keep:
    p = os.path.join(src, f)
    if not os.path.exists(p):
        print(f'{f}: not produced'); continue
    print(f'--- {f} ---')
    print(json.dumps(json.load(open(p)), indent=1)[:1500])
    if dst:
        shutil.copy(p, dst); print(f'  saved -> {dst}/{f}')